In [1]:
import pandas as pd
import numpy as np

In [10]:
%pip install pyarrow fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 692.1/692.1 kB 5.9 MB/s  0:00:00? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 5.4 MB/s  0:00:00m 7.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [fastparquet]
Note: you may need to restart the kernel to use updated packages.


In [2]:
stop_df = pd.read_csv("route_stops.csv")
df_routes = pd.read_csv("routes.csv")

In [3]:
# we sort the data by date.
df_routes = df_routes.sort_values(by='departure_planned')

In [4]:
# we are drop this column because this column It leads to data leakage.
cols_to_drop = ['actual_duration_min', 'on_time_delivery_rate', 'overall_delay_factor']
df_ml = df_routes.drop(columns=cols_to_drop)

In [5]:
# multicollinearity cleaning.
df_ml = df_ml.drop(columns=['road_incident'])

In [6]:
# we drop outliers. we want have 500 minutes on top.
p95_limit = df_ml['total_delay_min'].quantile(0.95)
df_ml['total_delay_min'] = np.clip(df_ml['total_delay_min'], a_min=None, a_max=p95_limit)

In [7]:
# one hot encoding time.
df_ml = pd.get_dummies(df_ml, columns=['weather_condition', 'traffic_level'], drop_first=True)

In [8]:
# only numerical values can be selected for x and y values.
y = df_ml['total_delay_min']
X = df_ml.drop(columns=[
    'total_delay_min', 
    'route_id', 
    'vehicle_id', 
    'driver_id', 
    'departure_planned', 
    'departure_actual'
])

In [9]:
print("X Matrix Size:", X.shape)
display(X.head())

X Matrix Size: (200, 18)


,vehicle_type,num_stops,total_distance_km,planned_duration_min,temperature_c,precipitation_mm,wind_speed_kmh,humidity_pct,visibility_km,incident_severity,weather_condition_cloudy,weather_condition_fog,weather_condition_rain,weather_condition_snow,weather_condition_wind,traffic_level_high,traffic_level_low,traffic_level_moderate
41,truck,10,339.14,574.9,25.7,0.0,13.6,91.4,12.8,0.0,False,False,False,False,True,False,True,False
21,car,10,187.59,252.7,15.1,0.0,13.2,80.9,24.1,0.0,False,False,False,False,False,True,False,False
137,van,6,142.37,250.9,11.4,0.0,3.8,50.1,22.8,0.0,False,False,False,False,True,False,False,False
173,truck,8,250.00,261.9,14.3,6.6,30.5,64.9,7.4,0.0,False,False,False,True,False,False,False,False
126,motorcycle,4,178.84,312.2,21.3,0.0,12.1,65.8,24.5,0.0,False,False,False,False,False,True,False,False


In [10]:
# we have to doing label encoding vehicle_type
X = pd.get_dummies(X, columns=['vehicle_type'], drop_first=True)

# clean data.
X.to_csv('X_clean.csv', index=False)
y.to_csv('y_target.csv', index=False)
print("Matrix Size:", X.shape)

Matrix Size: (200, 20)


In [11]:
# we are saving x and y file.
X.to_parquet('X_processed.parquet', index=False)
y.to_frame().to_parquet('y_processed.parquet', index=False)

print("Clean data is saved 'X_processed.parquet', y_processed.parquet")

Clean data is saved 'X_processed.parquet', y_processed.parquet
